# s11 — TodoWrite planning

**What this teaches:** the lightweight `todo_write` / `todo_read` / `todo_update` toolset — a scratch list the model uses to draft a plan before it touches the filesystem. The store persists the plan to `logs/agent_todo_<u>_<ts>.json`, so you can inspect (or replay) what the model thought it was going to do.

**Concept anchor:** todos are the model's working memory of *intent*. For durable cross-run plans with dependencies, look at [s12_tasks](../s12_tasks/s12_tasks.ipynb).

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [s01_loop](../s01_loop/s01_loop.ipynb)).
- A writable `logs/` directory in the kernel's cwd (auto-created if missing).

## 1. Imports

`todo` is the scratch-list package; `fstools` is appended so the agent can both *plan* (todo) and *execute* (read/write/bash) inside the same loop.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    fstools "github.com/blouargant/yoke/core/tools"
    "github.com/blouargant/yoke/internal/todo"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Create the todo store

`todo.NewStore("")` builds a session-scoped store. The empty string means "use the default user id"; pass a real id to isolate per-user state. The store creates `logs/agent_todo_<u>_<ts>.json` on first write — try `ls logs/` after running this notebook to inspect the JSON.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
store := todo.NewStore("")
fmt.Println("todo store ready")

## 4. Merge todo tools with filesystem tools

`store.Tools()` returns three tools:
- `todo_write` — overwrite the entire list with a fresh plan,
- `todo_read` — re-read the current plan (useful mid-loop),
- `todo_update` — flip individual items between `pending`/`in_progress`/`completed`.

Appending them to `fstools.New()` gives the model both planning and execution capabilities. The instruction nudges the model to plan *first*.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s11_todo",
    Description: "TodoWrite-first planning demo.",
    Model:       llm,
    Tools:       append(fstools.New(), store.Tools()...),
    Instruction: "ALWAYS call todo_write with a 3-5 step plan before doing anything else.",
})
must(err)
r, err := agentkit.Runner("s11", a)
must(err)
fmt.Println("agent + runner ready")

## 5. Run — plan, then execute

Watch the stream: the first tool call should be `todo_write`. After that the model may call `todo_update` to mark items in progress, then dispatch the actual `write` calls. Each item flip is persisted to the JSON file immediately.

In [ ]:
prompt := "Plan and then execute: create a file plan.md with three bullet points about adding TLS to a web server."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The first tool call should be `todo_write`. Subsequent flips through `todo_update` mirror real progress; the JSON on disk is the source of truth.
- For dependency-aware, *durable* plans that survive a kernel restart, prefer [s12_tasks](../s12_tasks/s12_tasks.ipynb) — its file format is richer and intentionally long-lived.

## Try it yourself

1. Open `logs/agent_todo_*_*.json` after the run to inspect the final state — items, statuses, ordering.
2. Re-run the cell with a different prompt and confirm the model overwrites the list (todo is *intent for this turn*, not a long-running journal).